# les codes des modèles

# DTW

In [ ]:

import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import seaborn as sns



#  Chargement des données
# > On importe  les fichiers CSV de mouvements / coordonnées X, Y, Z, T pour les regrouper dans un unique tableau de données global nommée " df" ici
df = pd.DataFrame(columns=['<x>', '<y>', '<z>', '<t>', 'gesture', 'subject', 'iter'])


for subject in range(1, 11):
  
    for number in range(0, 10):
 
        for iteration in range(1, 10+1):
         
            filename = f'Domain1_csv/Subject{subject}-{number}-{iteration}.csv'
      
            file = pd.read_csv(filename)
           
            file['gesture'] = number
       
            file['subject'] = subject
    
            file['iter'] = iteration
           
            df = pd.concat([df, file])


df.columns = ['x', 'y', 'z', 't', 'gesture', 'subject', 'iter']

print(df)


# Analyse exploratoire
# > On réalise des analyses sur les données pour mieux les comprendre et donc à chaque fois , on les affiche avec des graphiques


grouped = df.groupby(['gesture', 'iter'])

sequence_lengths = grouped.size().reset_index(name='length')

mean_lengths = sequence_lengths.groupby('gesture')['length'].mean().reset_index()

mean_lengths.columns = ['Chiffre', 'Frames moyennes']


plt.figure(figsize=(8, 5))

plt.bar(mean_lengths["Chiffre"], mean_lengths["Frames moyennes"])

plt.title("Nombre moyen de coordonnées par chiffre")

plt.xlabel("Chiffre")

plt.ylabel("Nombre moyen de coordonées")

plt.xticks(mean_lengths["Chiffre"])

plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()

plt.show()


durations = df.groupby(['gesture', 'iter'])['t'].agg(lambda x: x.max() - x.min()).reset_index(name='duration')

mean_durations = durations.groupby('gesture')['duration'].mean()

for gesture, mean_dur in mean_durations.items():
    print(f"Chiffre {gesture} : {mean_dur:.2f} s")


# Standardisation 
# > On crée une fonction pour centrer et réduire les coordonnées d'un groupe afin de rendre les tracés indépendants de leur taille ou position.
def standardize_gesture(df_group):

    df = df_group.copy()
  
    for axis in ['x', 'y', 'z']:
        df[f'{axis}_std'] = (df[axis] - df[axis].mean()) / df[axis].std()
  
    return df

df = df.groupby(['subject', 'gesture', 'iter'], group_keys=False).apply(standardize_gesture)

df.to_csv('df.csv', index=False)


# Rééchantillonnage à longueur fixe
# > On force tous les mouvements à avoir  le même nombre de points 100 ici
def resample_gesture(df, target_len=100):
 
    resampled_data = []

 
    for (gesture, subject, iteration), group in df.groupby(['gesture', 'subject', 'iter']):

        n = len(group)
  
        original_idx = np.arange(n)
    
        target_idx = np.linspace(0, n - 1, target_len)

        interp = lambda col: interp1d(original_idx, group[col], kind='linear')(target_idx)

    
        resampled_data.append(pd.DataFrame({
            'gesture': [gesture] * target_len,
            'subject': [subject] * target_len,
            'iter': [iteration] * target_len,
            'x_std': interp('x_std'),
            'y_std': interp('y_std'),
            'z_std': interp('z_std')
        }))


    return pd.concat(resampled_data, ignore_index=True)


df100 = resample_gesture(df)





#  Dictionnaires de découpage (Train / Test)
# > on Sépare des données en dictionnaires d'apprentissage et de test ici on selecitonnera que le dicitonnaire dict creation userout car on ne test que en mode " user independante" et donc cette fonction permet de determiner la phase " user independante"

def dict_creation_iterout(df, iter_out):
 
    test_df = df[df["iter"] == iter_out]

    train_df = df[df["iter"] != iter_out]

  
    dict_train = defaultdict(list)
    dict_test = defaultdict(list)


    for (gesture, subject, iteration), group in train_df.groupby(["gesture", "subject", "iter"]):
        dict_train[gesture].append(group[["x_std", "y_std", "z_std"]].values.tolist())

    for (gesture, subject, iteration), group in test_df.groupby(["gesture", "subject", "iter"]):
        dict_test[gesture].append(group[["x_std", "y_std", "z_std"]].values.tolist())


    return dict_train, dict_test


def dict_creation_userout(df, subj_out):

    test_df = df[df["subject"] == subj_out]

    train_df = df[df["subject"] != subj_out]

 
    dict_train = defaultdict(list)
    dict_test = defaultdict(list)


    for (gesture, subject, iteration), group in train_df.groupby(["gesture", "subject", "iter"]):
        dict_train[gesture].append(group[["x_std", "y_std", "z_std"]].values.tolist())


    for (gesture, subject, iteration), group in test_df.groupby(["gesture", "subject", "iter"]):
        dict_test[gesture].append(group[["x_std", "y_std", "z_std"]].values.tolist())

 
    return dict_train, dict_test




#  Fonctions de calcul des distances
 ##  > On réalise la distance Euclidienne et le DTW 
def eucl_dist(p1, p2):
    return np.sqrt((p1[0] - p2[0])**2 +
                   (p1[1] - p2[1])**2 +
                   (p1[2] - p2[2])**2)



def dtw_dist(seq1, seq2):

    n, m = len(seq1), len(seq2)
 
    dtw = np.full((n+1, m+1), np.inf)

    dtw[0, 0] = 0


    for i in range(1, n+1):
        for j in range(1, m+1):
          
            cost = eucl_dist(seq1[i-1], seq2[j-1])
     
            dtw[i, j] = cost + min(dtw[i-1, j],
                                   dtw[i, j-1],
                                   dtw[i-1, j-1])

  
    return dtw[n, m]


# Algorithme de prédiction KNN-DTW 

# > On crée le KNN qui va chercher à deviner le chiffre d'un geste inconnu en regardant les exemples connus sur les plus proches voisin .
def knn_dtw_predict(reference_seq, train_dict, k=3):
  
    distances = []


    for gesture, sequences in train_dict.items():
        for seq in sequences:
        
            dist = dtw_dist(reference_seq, seq)
        
            distances.append((gesture, dist))


    distances.sort(key=lambda x: x[1])
  
    k_nearest = distances[:k]


    votes = Counter([g for g, _ in k_nearest])

    return votes.most_common(1)[0][0]




#  Phase de test , de validation du modèle et du calcule des scores
## > on exécute la validation croisée, le calcul du score de précision final et affichage graphique de la matrice de confusion qui en résulte

def testmodel(DictCreationFunc, df, loop_range=10, k=3):

    total_true = []
    total_pred = []

 
    for i in range(1, loop_range + 1):
    
        print(f"\n--- Test iter/user OUT : {i} ---")
     
        coord_train, coord_test = DictCreationFunc(df, i)

     
        y_true, y_pred = [], []

     
        for gesture, sequences in coord_test.items():
            for seq in sequences:
              
                pred = knn_dtw_predict(seq, coord_train, k=k)
            
                y_true.append(gesture)
             
                y_pred.append(pred)

    
        acc = accuracy_score(y_true, y_pred)
     
        total_true.extend(y_true)
    
        total_pred.extend(y_pred)
     
        print(f"Exactitude split {i} : {acc:.2%}")


    final_acc = accuracy_score(total_true, total_pred)
  
    print(f"\nExactitude globale KNN-DTW : {final_acc:.2%}")


    cm = confusion_matrix(total_true, total_pred)
  
    disp = ConfusionMatrixDisplay(cm)
  
    disp.plot(cmap="Blues")

    plt.title("Matrice de confusion – KNN + DTW")

    plt.show()


    return total_true, total_pred


#  Exécution 

# On lance le test final en mode "User-Out" donc "user independante" 
testmodel(dict_creation_userout, df100)


# edit distance

In [ ]:

import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d


##  Chargement et préparation : même que pour le dtw


df = pd.DataFrame(columns=['<x>', '<y>', '<z>', '<t>', 'gesture', 'subject', 'iter'])


for subject in range(1, 11):

    for number in range(0, 10):

        for iteration in range(1, 11):
          
            filename = f"Domain1_csv/Subject{subject}-{number}-{iteration}.csv"
        
            file = pd.read_csv(filename)
         
            file["gesture"] = number
       
            file["subject"] = subject
        
            file["iter"] = iteration
        
            df = pd.concat([df, file])


df.columns = ['x', 'y', 'z', 't', 'gesture', 'subject', 'iter']


#  Standardisation : idem que dtw

def standardize_gesture(df_group):
   
    df = df_group.copy()

    for axis in ['x', 'y', 'z']:
        df[f"{axis}_std"] = (df[axis] - df[axis].mean()) / df[axis].std()
 
    return df


df = df.groupby(["subject", "gesture", "iter"], group_keys=False).apply(standardize_gesture)


#  Rééchantillonnage à longueur fixe : idem qie dtw

def resample_gesture(df, target_len=100):

 
    resampled_data = []

 
    for (gesture, subject, iteration), group in df.groupby(["gesture", "subject", "iter"]):

      
        n = len(group)
      
        original_idx = np.arange(n)
    
        target_idx = np.linspace(0, n - 1, target_len)

 
        interp = lambda col: interp1d(original_idx, group[col], kind='linear')(target_idx)

   
        resampled_data.append(pd.DataFrame({
            "gesture": [gesture]*target_len,
            "subject": [subject]*target_len,
            "iter": [iteration]*target_len,
            "x_std": interp("x_std"),
            "y_std": interp("y_std"),
            "z_std": interp("z_std")
        }))

   
    return pd.concat(resampled_data, ignore_index=True)


df100 = resample_gesture(df)


#  Séquences de symboles
# On regroupement et fusionne des lettres générées pour chaque point afin d'assembler les trajectoires sous forme de mots textuels continus. C'est a dire que l'on rassemble les lettres pour en faire des séquences


def gesture_to_sequence(df):

    seq_dict = defaultdict(list)

    for (gesture, subject, iteration), group in df.groupby(["gesture", "subject", "iter"]):
 
        seq = "".join(group["symbol"].tolist())
  
        seq_dict[gesture].append(seq)

    return seq_dict


#  Dictionnaires de découpage (Train / Test) : idem qie DTW


def dict_creation_iterout(df, iter_out):

    test_df = df[df["iter"] == iter_out]

    train_df = df[df["iter"] != iter_out]

    return train_df, test_df


def dict_creation_userout(df, subj_out):

    test_df = df[df["subject"] == subj_out]

    train_df = df[df["subject"] != subj_out]

    return train_df, test_df


#  Clustering  et conversion
# > on crée les clusters avec le Kmeans et on convertit chaque coordonnée 3D en une lettre de l'alphabet.


def cluster_and_convert(df_train, df_test, k_clusters=50):

  
    train_points = df_train[['x_std', 'y_std', 'z_std']].values
  
    kmeans = KMeans(n_clusters=k_clusters, random_state=42)

    kmeans.fit(train_points)


    def assign_symbols(df_local):
     
        pts = df_local[['x_std', 'y_std', 'z_std']].values
    
        clusters = kmeans.predict(pts)
     
        symbols = [chr(ord('A') + (c % 26)) for c in clusters]
     
        df_local = df_local.copy()
   
        df_local["symbol"] = symbols

        return df_local

 
    train_sym = assign_symbols(df_train)
 
    test_sym = assign_symbols(df_test)

 
    return gesture_to_sequence(train_sym), gesture_to_sequence(test_sym)


#  Edit Distance
# > on implémente l'edit distance dont sa programmation dynamique


def edit_distance(s1, s2):

    n, m = len(s1), len(s2)

    M = [[0]*(m+1) for _ in range(n+1)]


    for i in range(n+1): M[i][0] = i
  
    for j in range(m+1): M[0][j] = j


    for i in range(1, n+1):
        for j in range(1, m+1):
        
            cost = 0 if s1[i-1] == s2[j-1] else 1
       
            M[i][j] = min(
                M[i-1][j] + 1,
                M[i][j-1] + 1,
                M[i-1][j-1] + cost
            )

    return M[n][m]


# 8. KNN : idem que le dtw 


def knn_edit_predict(seq, train_dict, k=7):

  
    distances = []

    for gesture, sequences in train_dict.items():
        for s in sequences:
         
            d = edit_distance(seq, s)
    
            distances.append((gesture, d))

   
    distances.sort(key=lambda x: x[1])
 
    k_nearest = distances[:k]

    votes = Counter([g for g, _ in k_nearest])

  
    return votes.most_common(1)[0][0]


#  Phase de test , de validation du modèle et du calcule des scores : idem que dtw


def testmodel(DictCreationFunc, df, loop_range=10, k=7, k_clusters=50):

  
    total_true = []
    total_pred = []


    for i in range(1, loop_range+1):

   
        print(f"\n=== TEST OUT : {i} ===")


        train_df, test_df = DictCreationFunc(df, i)

     
        coord_train, coord_test = cluster_and_convert(train_df, test_df, k_clusters)


        y_true, y_pred = [], []


        for gesture, sequences in coord_test.items():
            for seq in sequences:
       
                pred = knn_edit_predict(seq, coord_train, k=k)
             
                y_true.append(gesture)
               
                y_pred.append(pred)

        acc = accuracy_score(y_true, y_pred)
  
        print(f"Exactitude split {i} : {acc:.2%}")

      
        total_true.extend(y_true)
       
        total_pred.extend(y_pred)

    final_acc = accuracy_score(total_true, total_pred)

    print(f"\n=== EXACTITUDE GLOBALE : {final_acc:.2%} ===")

 
    cm = confusion_matrix(total_true, total_pred)

    disp = ConfusionMatrixDisplay(cm)
 
    disp.plot(cmap="Blues")
 
    plt.title("Matrice de confusion – KNN + EditDistance")

    plt.show()


    return total_true, total_pred


#  Exécution : idem


testmodel(dict_creation_userout, df100)


# Longest common subsequence ( LCSQ)

In [ ]:

import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

##  Chargement et préparation : même que pour le dtw



df = pd.DataFrame(columns=['x', 'y', 'z', 't', 'gesture', 'subject', 'iter'])

for subject in range(1, 11):

    for number in range(0, 10):
 
        for iteration in range(1, 11):
    
            filename = f'Domain1_csv/Subject{subject}-{number}-{iteration}.csv'
        
            try:
           
                file = pd.read_csv(filename)
           
                file['gesture'] = number
   
                file['subject'] = subject
       
                file['iter'] = iteration
           
                file.columns = ['x', 'y', 'z', 't', 'gesture', 'subject', 'iter']
             
                df = pd.concat([df, file])
         
            except FileNotFoundError:
                continue

#  Standardisation : idem que dtw
def standardize_gesture(df_group):
  
    res = df_group.copy()

    for axis in ['x', 'y', 'z']:
        res[f'{axis}_std'] = (res[axis] - res[axis].mean()) / res[axis].std()
 
    return res


df = df.groupby(['subject', 'gesture', 'iter'], group_keys=False).apply(standardize_gesture)

#  Rééchantillonnage à longueur fixe : idem qie dtw
def resample_gesture(df_input, target_len=100):

    resampled_data = []

    for (gesture, subject, iteration), group in df_input.groupby(['gesture', 'subject', 'iter']):
     
        n = len(group)
     
        if n < 2: continue
 
        original_idx = np.arange(n)
    
        target_idx = np.linspace(0, n - 1, target_len)
   
        interp = lambda col: interp1d(original_idx, group[col], kind='linear')(target_idx)

    
        resampled_data.append(pd.DataFrame({
            'gesture': [gesture] * target_len,
            'subject': [subject] * target_len,
            'iter': [iteration] * target_len,
            'x_std': interp('x_std'),
            'y_std': interp('y_std'),
            'z_std': interp('z_std')
        }))
 
    return pd.concat(resampled_data, ignore_index=True)

df100 = resample_gesture(df)

#  Clustering et conversion : idem edit distance


def cluster_and_symbolize(train_df, test_df, k_clusters=20):
  
    kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)

    train_points = train_df[['x_std', 'y_std', 'z_std']].values

    kmeans.fit(train_points)

  
    def to_seq_dict(df_target):

        pts = df_target[['x_std', 'y_std', 'z_std']].values
      
        clusters = kmeans.predict(pts)
      
        symbols = [chr(ord('A') + (c % 26)) for c in clusters]

     
        temp_df = df_target.copy()

        temp_df['symbol'] = symbols

     
        seq_dict = defaultdict(list)
       
        for (gest, subj, it), group in temp_df.groupby(['gesture', 'subject', 'iter']):
         
            seq_dict[gest].append("".join(group['symbol'].tolist()))

        return seq_dict

  
    return to_seq_dict(train_df), to_seq_dict(test_df)

#  LCSQ
# > on implémente le LCSQ dont sa programmation dynamique et puis on convertit la valeur finale de la matrice en distance avec la formule 



def lcs_length(s1, s2):
 
    n, m = len(s1), len(s2)
 
    dp = [[0]*(m+1) for _ in range(n+1)]
 
    for i in range(1, n+1):
        for j in range(1, m+1):
        
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
         
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    return dp[n][m]


def lcs_distance(s1, s2):
    return len(s1) + len(s2) - 2 * lcs_length(s1, s2)

# 6. KNN : idem que le dtw 

def knn_lcs_predict(seq, train_dict, k=3):
 
    distances = []
  
    for gesture, sequences in train_dict.items():
        for s in sequences:
      
            d = lcs_distance(seq, s)
         
            distances.append((gesture, d))

    distances.sort(key=lambda x: x[1])

    k_nearest = distances[:k]
 
    votes = Counter([g for g, _ in k_nearest])

    return votes.most_common(1)[0][0]

#  Dictionnaires de découpage (Train / Test) : idem qie DTW


def dict_creation_userout(df_in, subj_out):

    test_df = df_in[df_in["subject"] == subj_out]

    train_df = df_in[df_in["subject"] != subj_out]
   
    return train_df, test_df



#  Phase de test , de validation du modèle et du calcule des scores : idem que dtw
def testmodel(df_input, loop_range=10, k_knn=3, k_clusters=20):

    total_true, total_pred = [], []


    for i in range(1, loop_range + 1):
   
        print(f"\n--- TEST SUJET SORTI : {i} ---")

      
        train_df, test_df = dict_creation_userout(df_input, i)
  
        if test_df.empty: continue

       
        train_dict, test_dict = cluster_and_symbolize(train_df, test_df, k_clusters)

     
        y_true, y_pred = [], []
    
        for gesture, sequences in test_dict.items():
            for seq in sequences:
              
                pred = knn_lcs_predict(seq, train_dict, k=k_knn)
           
                y_true.append(gesture)
        
                y_pred.append(pred)

     
        acc = accuracy_score(y_true, y_pred)
      
        print(f"Précision pour ce split : {acc:.2%}")
  
        total_true.extend(y_true)
 
        total_pred.extend(y_pred)

  
    print(f"\n=== EXACTITUDE GLOBALE : {accuracy_score(total_true, total_pred):.2%} ===")


    cm = confusion_matrix(total_true, total_pred)
 
    disp = ConfusionMatrixDisplay(cm)

    disp.plot(cmap="Blues")

    plt.title("Matrice de confusion – KNN + LCS (Clustering in Cross-Val)")
 
    plt.show()
#  Exécution : idem
testmodel(df100)

# lstm

In [ ]:

import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from tensorflow.keras import models, layers

## Chargement et préparation : même que pour le dtw



df = pd.DataFrame(columns=['<x>', '<y>', '<z>', '<t>', 'gesture', 'subject', 'iter'])


for subject in range(1, 11):
 
    for number in range(0, 10):
  
        for iteration in range(1, 11):
           
            filename = f'Domain1_csv/Subject{subject}-{number}-{iteration}.csv'

            file = pd.read_csv(filename)
      
            file['gesture'] = number

            file['subject'] = subject

            file['iter'] = iteration
   
            df = pd.concat([df, file])


df.columns = ['x', 'y', 'z', 't', 'gesture', 'subject', 'iter']

print(df)

# Analyse exploratoire : idem DTW


grouped = df.groupby(['gesture', 'iter'])

sequence_lengths = grouped.size().reset_index(name='length')

mean_lengths = sequence_lengths.groupby('gesture')['length'].mean().reset_index()

mean_lengths.columns = ['Chiffre', 'Frames moyennes']


plt.figure(figsize=(8, 5))

plt.bar(mean_lengths["Chiffre"], mean_lengths["Frames moyennes"])

plt.title("Nombre moyen de coordonnées par chiffre")

plt.xlabel("Chiffre")

plt.ylabel("Nombre moyen de coordonnées")

plt.xticks(mean_lengths["Chiffre"])

plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()

plt.show()


durations = df.groupby(['gesture', 'iter'])['t'].agg(lambda x: x.max() - x.min()).reset_index(name='duration')

mean_durations = durations.groupby('gesture')['duration'].mean()

for gesture, mean_dur in mean_durations.items():
    print(f"Chiffre {gesture} : {mean_dur:.2f} s")

# Standardisation : idem que dtw



def standardize_gesture(df_group):
  
    df = df_group.copy()

    for axis in ['x', 'y', 'z']:
        df[f'{axis}_std'] = (df[axis] - df[axis].mean()) / df[axis].std()

    return df

df = df.groupby(['subject', 'gesture', 'iter'], group_keys=False).apply(standardize_gesture)

df.to_csv('df.csv', index=False)

#  Rééchantillonnage à longueur fixe : idem que dtw


def resample_gesture(df, target_len=100):

    resampled_data = []


    for (gesture, subject, iteration), group in df.groupby(['gesture', 'subject', 'iter']):
  
        n = len(group)
   
        original_idx = np.arange(n)
    
        target_idx = np.linspace(0, n - 1, target_len)
   
        interp = lambda col: interp1d(original_idx, group[col], kind='linear')(target_idx)

        resampled_data.append(pd.DataFrame({
            'gesture': [gesture] * target_len,
            'subject': [subject] * target_len,
            'iter': [iteration] * target_len,
            'x_std': interp('x_std'),
            'y_std': interp('y_std'),
            'z_std': interp('z_std')
        }))


    return pd.concat(resampled_data, ignore_index=True)

df100 = resample_gesture(df)


# Création train/test user-independent
# > on crée la situaiton de user independante où dans le dtw il s'agit d'un dictionnaire et lstm ici , d'une matrice 


def create_user_independent_split(df, subj_out):

    train_df = df[df['subject'] != subj_out]

    test_df = df[df['subject'] == subj_out]

  
    def df_to_array(df_in):
    
        X, y = [], []
    
        for (gesture, subject, iteration), group in df_in.groupby(['gesture', 'subject', 'iter']):
      
            X.append(group[['x_std', 'y_std', 'z_std']].values)
      
            y.append(gesture)
     
        return np.array(X), np.array(y)

    X_train, y_train = df_to_array(train_df)

    X_test, y_test = df_to_array(test_df)

    return X_train, y_train, X_test, y_test


# LSTM Model
# > on crée l'architecture du modèle LSTM



def create_lstm_model(input_shape, num_classes):

    model = models.Sequential([
      
        layers.LSTM(256, return_sequences=True, dropout=0.2, input_shape=input_shape),
   
        layers.LSTM(256, dropout=0.3),

        layers.Dense(64 , activation='relu'),

        layers.Dropout(0.2),
     
        layers.Dense(num_classes, activation='softmax')
    ])
   
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return model

# Phase de test , de validation du modèle et du calcule des scores : idem que dtw


def test_lstm_user_independent(df, loop_range=10, epochs=50, batch_size=32):

    total_true, total_pred = [], []


    for subj_out in range(1, loop_range + 1):
       
        print(f"\n--- Test subject OUT : {subj_out} ---")
 
        X_train, y_train, X_test, y_test = create_user_independent_split(df, subj_out)

  
        model = create_lstm_model(input_shape=(X_train.shape[1], X_train.shape[2]), num_classes=len(np.unique(y_train)))
     
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)

        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
     
        acc = accuracy_score(y_test, y_pred)
 
        print(f"Exactitude subject {subj_out} : {acc:.2%}")

  
        total_true.extend(y_test)
    
        total_pred.extend(y_pred)


    final_acc = accuracy_score(total_true, total_pred)
 
    print(f"\nExactitude globale LSTM : {final_acc:.2%}")


    cm = confusion_matrix(total_true, total_pred)

    disp = ConfusionMatrixDisplay(cm)
 
    disp.plot(cmap="Blues")

    plt.title("Matrice de confusion – LSTM")
 
    plt.show()

# Exécution : idem


test_lstm_user_independent(df100)

# transformer 

In [ ]:

import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from tensorflow.keras import layers, models
import tensorflow as tf



## Chargement et préparation : même que pour le dtw



df = pd.DataFrame(columns=['<x>', '<y>', '<z>', '<t>', 'gesture', 'subject', 'iter'])


for subject in range(1, 11):
  
    for number in range(0, 10):
      
        for iteration in range(1, 11):

            filename = f'Domain1_csv/Subject{subject}-{number}-{iteration}.csv'
         
            file = pd.read_csv(filename)
     
            file['gesture'] = number
         
            file['subject'] = subject
         
            file['iter'] = iteration

            df = pd.concat([df, file])

df.columns = ['x', 'y', 'z', 't', 'gesture', 'subject', 'iter']

print(df)


# Analyse exploratoire : idem DTW


grouped = df.groupby(['gesture', 'iter'])

sequence_lengths = grouped.size().reset_index(name='length')

mean_lengths = sequence_lengths.groupby('gesture')['length'].mean().reset_index()

mean_lengths.columns = ['Chiffre', 'Frames moyennes']


plt.figure(figsize=(8, 5))

plt.bar(mean_lengths["Chiffre"], mean_lengths["Frames moyennes"])

plt.title("Nombre moyen de coordonnées par chiffre")

plt.xlabel("Chiffre")

plt.ylabel("Nombre moyen de coordonnées")

plt.xticks(mean_lengths["Chiffre"])

plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()

plt.show()


durations = df.groupby(['gesture', 'iter'])['t'].agg(lambda x: x.max() - x.min()).reset_index(name='duration')

mean_durations = durations.groupby('gesture')['duration'].mean()

for gesture, mean_dur in mean_durations.items():
    print(f"Chiffre {gesture} : {mean_dur:.2f} s")


# Standardisation : idem que dtw


def standardize_gesture(df_group):

    df = df_group.copy()

    for axis in ['x', 'y', 'z']:
        df[f'{axis}_std'] = (df[axis] - df[axis].mean()) / df[axis].std()
  
    return df


df = df.groupby(['subject', 'gesture', 'iter'], group_keys=False).apply(standardize_gesture)

df.to_csv('df.csv', index=False)


#  Rééchantillonnage à longueur fixe : idem qie dtw



def resample_gesture(df, target_len=100):

    resampled_data = []


    for (gesture, subject, iteration), group in df.groupby(['gesture', 'subject', 'iter']):
   
        n = len(group)
     
        original_idx = np.arange(n)
     
        target_idx = np.linspace(0, n - 1, target_len)
     
        interp = lambda col: interp1d(original_idx, group[col], kind='linear')(target_idx)


        resampled_data.append(pd.DataFrame({
            'gesture': [gesture] * target_len,
            'subject': [subject] * target_len,
            'iter': [iteration] * target_len,
            'x_std': interp('x_std'),
            'y_std': interp('y_std'),
            'z_std': interp('z_std')
        }))

  
    return pd.concat(resampled_data, ignore_index=True)


df100 = resample_gesture(df)


# Création train/test user-independent : idem que lstm 

def create_user_independent_split(df, subj_out):
 
    train_df = df[df['subject'] != subj_out]

    test_df = df[df['subject'] == subj_out]


    def df_to_array(df_in):
    
        X, y = [], []
   
        for (gesture, subject, iteration), group in df_in.groupby(['gesture', 'subject', 'iter']):
        
            X.append(group[['x_std', 'y_std', 'z_std']].values)
       
            y.append(gesture)
    
        return np.array(X), np.array(y)


    X_train, y_train = df_to_array(train_df)

    X_test, y_test = df_to_array(test_df)
  
    return X_train, y_train, X_test, y_test


# Mise en place du positional encoding
# > cette partie permet au trasnsformer de connaitre l'ordre des coordonnées  ( dans le temps ) car contrairement au lstm qui le connait , le trasnformer il faut lui donner cette information car ce modèle n'a pas cette infromation de base 


class PositionalEncoding(layers.Layer):
 
    def __init__(self, max_len, d_model):

        super().__init__()
      
        self.max_len = max_len
        self.d_model = d_model

    
        pos = np.arange(max_len)[:, np.newaxis]
   
        i = np.arange(d_model)[np.newaxis, :]
  
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))


        angle_rads = pos * angle_rates

       
        pe = np.zeros((max_len, d_model))
    
        pe[:, 0::2] = np.sin(angle_rads[:, 0::2])
    
        pe[:, 1::2] = np.cos(angle_rads[:, 1::2])

    
        self.positional_encoding = tf.constant(pe[np.newaxis, ...], dtype=tf.float32)


    def call(self, x):

        seq_len = tf.shape(x)[1]
      
        return x + self.positional_encoding[:, :seq_len, :]



# Transformer Encoder Model
# > on definit le bloc transformer encoder car on ne réalise que la partie encoder et pas la partie decoder du modèle


def transformer_encoder(x, num_heads, d_model, dff, dropout):

    attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(x, x)
 
    attn_output = layers.Dropout(dropout)(attn_output)

    out1 = layers.LayerNormalization(epsilon=1e-6)(x + attn_output)


    ffn = models.Sequential([
        layers.Dense(dff, activation="relu"),
        layers.Dense(d_model),
    ])
  
    ffn_output = ffn(out1)
 
    ffn_output = layers.Dropout(dropout)(ffn_output)
  
    out2 = layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output)

   
    return out2


# transformer Model
# > on crée l'architecture du modèle Transformer

def create_transformer_model(input_shape, num_classes, seq_len=100):
   
    inputs = layers.Input(shape=input_shape)

 
    d_model = 128
    num_heads = 8
    num_layers = 2
    dff = 256
    dropout = 0.2


    x = layers.Dense(d_model)(inputs)
   
    x = PositionalEncoding(seq_len, d_model)(x)


    for _ in range(num_layers):
        x = transformer_encoder(x, num_heads, d_model, dff, dropout)

   
    x = layers.GlobalAveragePooling1D()(x)
 
    x = layers.Dense(128, activation='relu')(x)

    x = layers.Dropout(0.2)(x)

 
    outputs = layers.Dense(num_classes, activation='softmax')(x)

  
    model = models.Model(inputs, outputs)
  
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return model


#  Phase de test , de validation du modèle et du calcule des scores : idem que dtw
def test_transformer_user_independent(df, loop_range=10, epochs=50, batch_size=32, verbose_fit=1):
  
    total_true, total_pred = [], []

    for subj_out in range(1, loop_range + 1):
   
        print(f"\n--- Test subject OUT : {subj_out} ---")

       
        X_train, y_train, X_test, y_test = create_user_independent_split(df, subj_out)


        n_classes = len(np.unique(y_train))
      
        input_shape = (X_train.shape[1], X_train.shape[2])


        model = create_transformer_model(input_shape=input_shape, num_classes=n_classes)


        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=verbose_fit)

    
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    
        acc = accuracy_score(y_test, y_pred)

        print(f"Exactitude subject {subj_out} : {acc:.2%}")


        total_true.extend(y_test)
     
        total_pred.extend(y_pred)


    final_acc = accuracy_score(total_true, total_pred)

    print(f"\nExactitude globale Transformer : {final_acc:.2%}")

    cm = confusion_matrix(total_true, total_pred)
 
    disp = ConfusionMatrixDisplay(cm)

    disp.plot(cmap="Blues")

    plt.title("Matrice de confusion – Transformer")

    plt.show()


# Exécution : idem
test_transformer_user_independent(df100, loop_range=10, epochs=50, batch_size=32, verbose_fit=1)